In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment and run this cell
# Local VS Code users: SKIP this cell
# ============================================================

# !pip install crewai crewai-tools duckduckgo-search python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# print("Colab setup complete!")

# Day 7: Advanced Multi-Agent Patterns

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Custom domain agents with industry-specific backstories
2. Explicit context passing between tasks (`context=[...]`)
3. Structured output with `output_json` (Pydantic models)
4. A competitive analysis crew — real-world business deliverable
5. Debugging multi-agent workflows (6-step checklist)
6. CrewAI memory for knowledge sharing

**Deliverable:** Domain-specific competitive analysis crew

---

## 0. Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'Loaded' if len(groq_key) > 10 else 'MISSING'}")

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

print(f"LLM configured: {llm.model}")

---
## 1. Custom Domain Agents — Industry-Specific Teams

Yesterday's agents were generic (Researcher, Writer, Editor). Today we design agents for **specific business domains**.

The secret: a domain-specific role + business-outcome goal + backstory with real companies and years of experience.

### 1.1 Financial Analysis Crew

In [ ]:
# Domain-specific agents with detailed backstories

market_analyst = Agent(
    role="Senior Market Research Analyst",
    goal="Identify market trends, competitive threats, and growth opportunities with data-backed insights",
    backstory=(
        "You have 15 years of experience at Goldman Sachs and McKinsey, specializing in "
        "emerging market analysis and competitive intelligence. You are known for your "
        "rigorous data-driven approach and ability to spot trends before they become obvious. "
        "You always quantify your findings with specific numbers and percentages."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

swot_analyst = Agent(
    role="Strategic SWOT Analyst",
    goal="Create comprehensive SWOT analyses that reveal actionable strategic insights",
    backstory=(
        "You are a strategic consultant who has conducted SWOT analyses for 200+ companies "
        "across India and Southeast Asia. You worked at Bain & Company for 8 years. "
        "Your SWOT analyses are known for being specific (not generic), actionable "
        "(each point leads to a clear action), and well-prioritized (most important first)."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

strategy_advisor = Agent(
    role="Chief Strategy Officer",
    goal="Synthesize research and analysis into clear, prioritized strategic recommendations",
    backstory=(
        "You led strategy at Infosys for 8 years and have advised 50+ companies on "
        "growth strategy, market entry, and competitive positioning. You think in terms of "
        "90-day action plans, resource allocation, and measurable KPIs. You always provide "
        "exactly 3-5 recommendations ranked by impact and feasibility."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("3 domain agents created:")
for agent in [market_analyst, swot_analyst, strategy_advisor]:
    print(f"  - {agent.role}")

**Notice the difference from Day 6:**
- Day 6: `role="Senior Research Analyst"` (generic)
- Day 7: `role="Senior Market Research Analyst"` with backstory mentioning Goldman Sachs, McKinsey, specific years

The LLM draws on its training data about how Goldman Sachs analysts think and write. More specific backstory = better output.

---

## 2. Context Passing — Explicit `context=[...]`

In sequential mode, each task only sees the task right before it. But what if the Strategy Advisor needs BOTH the market research AND the SWOT analysis?

The `context` parameter solves this:

In [ ]:
# Task 1: Market Research
research_task = Task(
    description=(
        "Research the competitive landscape for {company} in the {industry} industry. "
        "Identify their top 3 competitors, market share estimates, key differentiators, "
        "pricing strategies, and recent strategic moves. "
        "Focus on the Indian market and data from 2025-2026."
    ),
    expected_output=(
        "A structured competitive research report with: "
        "1) Company overview, 2) Top 3 competitors with comparison, "
        "3) Market share estimates, 4) Key trends affecting the industry."
    ),
    agent=market_analyst,
)

# Task 2: SWOT Analysis (receives research automatically in sequential mode)
swot_task = Task(
    description=(
        "Based on the competitive research provided, create a detailed SWOT analysis for {company}. "
        "Each category (S, W, O, T) should have 3-4 specific, actionable points. "
        "Avoid generic statements — every point should be backed by data from the research."
    ),
    expected_output=(
        "A SWOT analysis with 3-4 specific points per category, "
        "each tied to data from the competitive research."
    ),
    agent=swot_analyst,
)

# Task 3: Strategy — receives BOTH research AND SWOT via context=[]
strategy_task = Task(
    description=(
        "Based on the competitive research AND the SWOT analysis, create a strategic "
        "recommendation document for {company}. Provide exactly 3-5 recommendations "
        "ranked by impact. Each recommendation should include: what to do, why it matters, "
        "timeline (30/60/90 days), and how to measure success (KPIs)."
    ),
    expected_output=(
        "A strategic recommendation document with 3-5 prioritized recommendations, "
        "each with action, rationale, timeline, and KPIs."
    ),
    agent=strategy_advisor,
    context=[research_task, swot_task],  # receives BOTH outputs explicitly
)

print("3 tasks created with context passing:")
print(f"  Task 1 (Research): no explicit context (first task)")
print(f"  Task 2 (SWOT): receives Task 1 output automatically")
print(f"  Task 3 (Strategy): context=[research_task, swot_task] — sees BOTH")

**Why `context=[research_task, swot_task]` matters:**

Without it, the Strategy Advisor would only see the SWOT output (the task right before it). It would miss the raw competitive data from the Market Researcher. With `context`, it gets the full picture — research data AND strategic analysis.

---

## 3. Running the Competitive Analysis Crew

In [ ]:
# Assemble and run the crew
competitive_crew = Crew(
    agents=[market_analyst, swot_analyst, strategy_advisor],
    tasks=[research_task, swot_task, strategy_task],
    process=Process.sequential,
    verbose=True,
)

# Run it — this makes 3+ LLM calls
result = competitive_crew.kickoff(inputs={
    "company": "Zerodha",
    "industry": "online stock broking"
})

print("\n" + "=" * 60)
print("FINAL STRATEGIC RECOMMENDATIONS")
print("=" * 60)
print(result.raw)

In [ ]:
# Inspect individual task outputs — useful for debugging
if hasattr(result, 'tasks_output'):
    agents = [market_analyst, swot_analyst, strategy_advisor]
    for i, task_output in enumerate(result.tasks_output):
        print(f"\n{'='*60}")
        print(f"Task {i+1}: {agents[i].role}")
        print(f"{'='*60}")
        print(str(task_output)[:500], "...")

---
## 4. Structured Output with `output_json`

You can force a task to return validated JSON using a Pydantic model — similar to Day 4's PydanticOutputParser but built into CrewAI.

In [ ]:
from pydantic import BaseModel
from typing import List
import json

# Define the exact structure we want
class SWOTResult(BaseModel):
    company: str
    strengths: List[str]
    weaknesses: List[str]
    opportunities: List[str]
    threats: List[str]

# Create a task that returns structured JSON
structured_swot_task = Task(
    description=(
        "Create a SWOT analysis for {company} in the {industry} industry. "
        "Provide 3 specific points for each category."
    ),
    expected_output="A SWOT analysis with exactly 3 points per category.",
    agent=swot_analyst,
    output_json=SWOTResult,  # forces structured JSON output
)

# Run it with a simple crew
swot_crew = Crew(
    agents=[swot_analyst],
    tasks=[structured_swot_task],
    process=Process.sequential,
    verbose=True,
)

swot_result = swot_crew.kickoff(inputs={
    "company": "Flipkart",
    "industry": "e-commerce"
})

print("\n=== Structured SWOT Output ===")
print(f"Type: {type(swot_result.json_dict).__name__}")
print(json.dumps(swot_result.json_dict, indent=2))

**Why this matters:** In production, you need predictable output structure. `output_json` ensures you get a Python dict with exactly the fields you defined — no surprises.

---

## 5. Adding Web Search to the Competitive Crew

Let's give the Market Researcher actual web search capability for real-time competitor data.

In [ ]:
from crewai.tools import tool

@tool("Web Search")
def search_web(query: str) -> str:
    """Searches the web for current information. Use for finding competitor data,
    market statistics, recent news, and industry trends.
    
    Args:
        query: Search query, e.g. 'Zerodha competitors India 2026'
    """
    try:
        from duckduckgo_search import DDGS
        results = DDGS().text(query, max_results=3)
        if not results:
            return f"No results for '{query}'. Try different keywords."
        output = []
        for i, r in enumerate(results, 1):
            output.append(f"{i}. {r['title']}: {r['body'][:200]}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Search error: {e}"

# Recreate the market analyst with search
market_analyst_v2 = Agent(
    role="Senior Market Research Analyst",
    goal="Find comprehensive competitive intelligence using web search and analysis",
    backstory=(
        "You have 15 years at Goldman Sachs specializing in competitive intelligence. "
        "You use web search to find the latest data and always verify from multiple sources."
    ),
    tools=[search_web],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# Recreate tasks for the enhanced crew
research_task_v2 = Task(
    description=(
        "Research the competitive landscape for {company} in the {industry} industry. "
        "Use web search to find current competitor info, market share, and recent news. "
        "Search at least 2-3 times with different queries for thorough coverage."
    ),
    expected_output="A data-backed competitive report with findings from web search.",
    agent=market_analyst_v2,
)

swot_task_v2 = Task(
    description="Create a SWOT analysis based on the competitive research. 3-4 specific points each.",
    expected_output="A detailed SWOT analysis tied to research data.",
    agent=swot_analyst,
)

strategy_task_v2 = Task(
    description=(
        "Create 3-5 strategic recommendations ranked by impact. "
        "Include: what to do, why, timeline (30/60/90 days), and KPIs."
    ),
    expected_output="3-5 prioritized recommendations with timelines and KPIs.",
    agent=strategy_advisor,
    context=[research_task_v2, swot_task_v2],
)

enhanced_crew = Crew(
    agents=[market_analyst_v2, swot_analyst, strategy_advisor],
    tasks=[research_task_v2, swot_task_v2, strategy_task_v2],
    process=Process.sequential,
    verbose=True,
)

print(f"Enhanced crew ready! Researcher has: {[t.name for t in market_analyst_v2.tools]}")

In [ ]:
# Run the enhanced crew with real web search
result_v2 = enhanced_crew.kickoff(inputs={
    "company": "Swiggy",
    "industry": "food delivery"
})

print("\n" + "=" * 60)
print("STRATEGIC RECOMMENDATIONS (with real web data)")
print("=" * 60)
print(result_v2.raw)

---
## 6. Debugging Multi-Agent Workflows

When output is wrong, use this 6-step checklist:

| Step | Question | If yes, fix... |
|---|---|---|
| 1 | Which agent's output is wrong? | Check `result.tasks_output` per agent |
| 2 | Is it a role/backstory problem? | Output off-topic → fix backstory |
| 3 | Is it a task description problem? | Right role, wrong output → add detail to description |
| 4 | Is it a tool problem? | Tool error → fix tool code or docstring |
| 5 | Is it a context problem? | Missing info → check task order / `context=[]` |
| 6 | Is it a model problem? | 8B failing → switch to 70B for that agent |

### 6.1 Debugging in practice — inspecting each agent's output

In [ ]:
# How to debug: inspect each task's output individually
if hasattr(result_v2, 'tasks_output'):
    all_agents = [market_analyst_v2, swot_analyst, strategy_advisor]
    for i, task_output in enumerate(result_v2.tasks_output):
        output_text = str(task_output)
        word_count = len(output_text.split())
        print(f"Agent {i+1}: {all_agents[i].role}")
        print(f"  Output length: {word_count} words")
        print(f"  First 100 chars: {output_text[:100]}...")
        print()

# Debugging questions to ask:
# - Is the research thorough? (Agent 1)
# - Is the SWOT specific or generic? (Agent 2)
# - Are recommendations actionable with KPIs? (Agent 3)

---
## 7. CrewAI Memory — Shared Knowledge

Enable memory for richer context sharing across tasks within one crew run:

In [ ]:
# Create a crew with memory enabled
memory_crew = Crew(
    agents=[market_analyst_v2, swot_analyst, strategy_advisor],
    tasks=[research_task_v2, swot_task_v2, strategy_task_v2],
    process=Process.sequential,
    verbose=True,
    memory=True,  # enables shared memory across tasks
)

print("Memory-enabled crew created!")
print("Memory allows agents to access a shared knowledge base within this crew run.")
print("This is different from Day 3 memory (conversation turns) — this is task-to-task sharing.")

**Note:** Memory in CrewAI is within-run only — it doesn't persist between `kickoff()` calls. For persistent memory across sessions, you'll combine RAG + agents in Marathon 2.

---

## 8. Different Domain — Same Pattern

The competitive analysis pattern works for any industry. Just change the inputs:

In [ ]:
# Same crew, completely different domain
result_v3 = enhanced_crew.kickoff(inputs={
    "company": "BYJU'S",
    "industry": "edtech"
})

print("\n" + "=" * 60)
print("STRATEGIC RECOMMENDATIONS — BYJU'S")
print("=" * 60)
print(result_v3.raw)

---
## 9. Decision Framework — When to Use What

| Your need | Use this |
|---|---|
| Simple question, one step | Single LLM call (Day 1) |
| Needs tools or external data | Single agent — LangChain (Days 4-5) |
| Multiple steps, clear order | Multi-agent — CrewAI sequential (Day 6) |
| Complex, domain-specific, needs context passing | CrewAI advanced (Day 7 — today) |
| Needs loops, conditionals, state management | LangGraph (tomorrow!) |

---

## 10. Your Turn — Build a Domain Crew

**Challenge (10 minutes):** Build a competitive analysis crew for a different domain.

**Ideas:**
- **Healthcare:** Analyze a hospital chain vs competitors (Apollo vs Fortis vs Max)
- **SaaS:** Analyze a software company (Freshworks vs Zoho vs Salesforce)
- **Retail:** Analyze a retail brand (Reliance Retail vs DMart vs BigBasket)
- **Banking:** Analyze a bank (HDFC vs ICICI vs SBI)

**Requirements:**
1. 3 domain agents with specific backstories
2. 3 tasks with context passing on the final task
3. At least one task with `output_json`
4. Web search tool on the researcher

In [ ]:
# YOUR CODE HERE

# Step 1: Create 3 domain agents
# researcher = Agent(role="...", goal="...", backstory="...", tools=[search_web], llm=llm, ...)
# analyst = Agent(role="...", goal="...", backstory="...", llm=llm, ...)
# advisor = Agent(role="...", goal="...", backstory="...", llm=llm, ...)

# Step 2: Create 3 tasks with context passing
# task1 = Task(description="...", expected_output="...", agent=researcher)
# task2 = Task(description="...", expected_output="...", agent=analyst)
# task3 = Task(description="...", expected_output="...", agent=advisor, context=[task1, task2])

# Step 3: Create and run crew
# crew = Crew(agents=[...], tasks=[...], process=Process.sequential, verbose=True)
# result = crew.kickoff(inputs={"company": "...", "industry": "..."})
# print(result.raw)

---
## 11. Day 7 Wrap-up

### What you built today:
- Custom domain agents with industry-specific backstories (Goldman Sachs, McKinsey, Bain)
- Explicit context passing with `context=[task1, task2]`
- Structured output with `output_json` and Pydantic models
- Competitive analysis crew — a real business deliverable
- Web search integrated into the researcher agent
- Debugging multi-agent systems (6-step checklist)
- CrewAI memory for task-to-task knowledge sharing

### Key patterns to remember:
```python
# Explicit context passing
task3 = Task(..., context=[task1, task2])  # sees both outputs

# Structured JSON output
task = Task(..., output_json=MyPydanticModel)

# Save to file
task = Task(..., output_file="report.md")

# Memory
crew = Crew(..., memory=True)
```

### The framework journey so far:
```
Smolagents (Days 1-2) → LangChain (Days 4-5) → CrewAI (Days 6-7) → LangGraph (tomorrow!)
  learn fast              build pipelines        multi-agent teams     stateful workflows
```

### Tomorrow — Day 8: LangGraph — Stateful Agent Workflows
Graphs, conditional routing, human-in-the-loop approval, checkpointing with MemorySaver. The most powerful agent pattern yet — and the last framework before capstone.

---

### Resources:
- [CrewAI docs](https://docs.crewai.com/)
- [CrewAI task context](https://docs.crewai.com/concepts/tasks)
- [CrewAI structured output](https://docs.crewai.com/concepts/tasks#output-json)